<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase34_COLAB_DOWNLOAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 — Phase 34
## Adaptive MRI evidence carrier: a *new algorithmic prototype* for source-to-shadow information transport
**Google Drive → Google Drive** · synthetic-only · follows verified Phase 33 · no Athena required.

### Scientific task
Build a compact evidence carrier that recovers a pre-specified set of neuroimaging dimensions discarded by an explicitly simulated value-only representation. Compare five deterministic carrier plans under the constrained objective **minimum serialized metadata bytes subject to exact field reconstruction**. Use held-out Phase 33 synthetic source records, but do **not** use its injected fault labels to design or run this reconstruction algorithm.

This is an **imaging-specific transport prototype that may inform future Arm D**. It is *not* an established new scientific method, not an execution of the original generic Arm C, not a standards-compliant Arm A, and not an official OMOP/MI-CDM target cohort experiment. The algorithm's novelty relative to the literature remains to be established. Phase 33's same-developer synthetic cohort is not independent clinical validation.

The 19 Athena binding decisions and 36 unresolved original vocabulary gaps remain unchanged. No concept IDs or real DICOM/procedure provenance are fabricated. **No real ADNI records are loaded or exported.**

In [1]:
import os
DRIVE_ROOT_OVERRIDE=os.environ.get('ICHI2027_DRIVE_ROOT','').strip()  # Local regression tests only
if not DRIVE_ROOT_OVERRIDE:
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime,timezone
import json,csv,hashlib,copy,re,os,math

ROOT=Path(DRIVE_ROOT_OVERRIDE) if DRIVE_ROOT_OVERRIDE else Path('/content/drive/MyDrive')
PREFERRED_PHASE33_DIR=''   # Optional exact folder, e.g. '/content/drive/MyDrive/NeuroFHIR_ICHI2027'.
EXPECTED_PHASE='33_person_disjoint_label_masked_synthetic_evidence_audit'
INPUT_NAMES={
    'bundle':'ICHI2027_Phase33_NEW_SYNTHETIC_FHIR_BUNDLE.json',
    'cases':'ICHI2027_Phase33_HELDOUT_CASE_RESULTS.csv',
    'cohort':'ICHI2027_Phase33_SOURCE_PROXY_COHORT.csv',
}
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def canonical(obj):
    return json.dumps(obj,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def byte_length(obj):
    return len(canonical(obj).encode('utf-8'))
def all_files_safe(base):
    """Google Drive may yield stale file entries; do not sort by unsafe stat()."""
    for root, dirs, files in os.walk(str(base),onerror=lambda _:None):
        dirs[:]=[d for d in dirs if d not in ['.git','_PRIVATE_ATHENA']]
        for name in sorted(files):
            yield Path(root)/name
def name_matches(actual,base):
    stem,ext=os.path.splitext(base)
    return (actual==base or bool(re.fullmatch(re.escape(stem)+r'\(\d+\)'+re.escape(ext),actual)))
def matching_sibling(folder,basename,expected):
    for p in all_files_safe(folder):
        if p.parent!=folder or not name_matches(p.name,basename):continue
        try:
            if sha(p)==expected:return p
        except (OSError,FileNotFoundError):continue
    return None

assert ROOT.is_dir(),f'Google Drive root missing: {ROOT}'
search_root=Path(PREFERRED_PHASE33_DIR) if PREFERRED_PHASE33_DIR else ROOT
candidates=[p for p in all_files_safe(search_root)
            if name_matches(p.name,'ICHI2027_Phase33_SHAREABLE.json')]
selected=None; failures=[]
for share in candidates:
    try:
        summary=json.loads(share.read_text(encoding='utf-8'))
        if summary.get('phase')!=EXPECTED_PHASE:continue
        checks=summary['artifact_sha256']
        found={k:matching_sibling(share.parent,n,checks[k]) for k,n in INPUT_NAMES.items()}
        if not all(found.values()):
            failures.append(f'{share.parent}: hash-matching Phase33 bundle/cases/cohort missing');continue
        selected=(share,summary,found);break
    except (OSError,ValueError,KeyError,FileNotFoundError) as exc:
        failures.append(f'{share}: {exc!r}')
if selected is None:
    raise FileNotFoundError('Could not find complete hash-matched Phase 33 inputs in one Drive folder. '
                            'Copy SHAREABLE, NEW_SYNTHETIC_FHIR_BUNDLE, HELDOUT_CASE_RESULTS, '
                            'SOURCE_PROXY_COHORT into the same folder. '+str(failures[:3]))
P33_PATH,p33,P33_FILES=selected
OUT_DIR=P33_PATH.parent
assert p33['evaluation']['case_count']==19 and p33['evaluation']['injected_events_tp']==24
assert p33['method']['official_omop_or_micdm_comparison'] is False
assert p33['upstream']['unreviewed_athena_gaps']==36
with P33_FILES['bundle'].open(encoding='utf-8') as f: BUNDLE=json.load(f)
assert BUNDLE['resourceType']=='Bundle'
def load_csv(path):
    with open(path,encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
P33_CASES=load_csv(P33_FILES['cases'])
P33_COHORT=load_csv(P33_FILES['cohort'])
assert len(P33_CASES)==19 and len(P33_COHORT)==19
print('Verified Phase33 input folder:',OUT_DIR)
print('Exact input SHA256:',{k:sha(v) for k,v in P33_FILES.items()})

Verified Phase33 input folder: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs
Exact input SHA256: {'bundle': '4d2e64b3dad926f9a32fc2d243f8ebb04dcff0159d60c5343f01e0a1985d1bf4', 'cases': '754315fad00a2cee1b7a2847063a90219db9b6f56b34b7f9b7d38bfaf9adb519', 'cohort': 'ab4024881c2d439c11a42b3ce618924a6a12d002994d34960f6f51d0b9127133'}


## 1. Source-only canonical evidence manifest
The source representation is built exclusively from Phase 33's **clean, synthetic FHIR bundle**. It preserves observation identities and 13 selected evidence dimensions: anatomical region and side, unit, study/procedure context that actually exists in source, synthetic study and series UIDs, imaging modality, device, provenance references, and the source's preexisting TRACE metadata. **Phase 33 injected challenge labels are not inputs to this algorithm.**

The *value-only shadow* deliberately retains an opaque, non-semantic observation key, person, timepoint, effective time, and numeric quantity. The original FHIR observation ID remains in the evidence carrier, not in the numeric base, so anatomy cannot be recovered by reading the ID string. This is a **simulated ablation**, not an exported MI-CDM or OMOP table.

In [3]:
RESOURCES=[e['resource'] for e in BUNDLE['entry']]
by_ref={f"{r.get('resourceType')}/{r.get('id')}":r for r in RESOURCES}
assert len(by_ref)==len(RESOURCES),'Duplicate resource type/id in input'
patients={k for k in by_ref if k.startswith('Patient/')}
studies={k:r for k,r in by_ref.items() if k.startswith('ImagingStudy/')}
obs={k:r for k,r in by_ref.items() if k.startswith('Observation/')}
prov={k:r for k,r in by_ref.items() if k.startswith('Provenance/')}
prov_by_target=defaultdict(list)
for ref,p in prov.items():
    for t in p.get('target',[]):prov_by_target[t.get('reference','')].append((ref,p))

def source_ext(o,tail):
    vals=[]
    for x in o.get('extension',[]):
        if str(x.get('url','')).endswith('/'+tail):
            vals.append(x.get('valueCode',x.get('valueString','')))
    if len(vals)!=1:raise ValueError(f'Expected exactly one {tail} extension: {o["id"]}')
    return str(vals[0])
def study_ids(s):
    found=[x['value'] for x in s.get('identifier',[])
           if x.get('system')=='urn:dicom:uid' and isinstance(x.get('value'),str)]
    if len(found)!=1 or not found[0].startswith('urn:oid:'):
        raise ValueError(f'Not exactly one synthetic study UID: {s.get("id")}')
    series=s.get('series',[])
    if len(series)!=1 or not series[0].get('uid'):
        raise ValueError(f'Ambiguous/missing study series: {s.get("id")}')
    uid=found[0][len('urn:oid:'):]
    def syntactic_uid(x):
        return len(x)<=64 and bool(re.fullmatch(r'[0-9]+(?:\.[0-9]+)+',x))
    assert syntactic_uid(uid) and syntactic_uid(series[0]['uid'])
    return uid,series[0]['uid'],series[0].get('modality',{}).get('code','')

def evidence_row(ref,o):
    person=o.get('subject',{}).get('reference','')
    if person not in patients:raise ValueError('Observation subject is not a Patient: '+ref)
    fs=[x.get('reference') for x in o.get('focus',[])]
    if len(fs)!=1 or fs[0] not in studies:raise ValueError('Unresolved or ambiguous imaging study '+ref)
    study_ref=fs[0];study=studies[study_ref]
    if study.get('subject',{}).get('reference')!=person:raise ValueError('Cross-patient study '+ref)
    study_uid,series_uid,modality=study_ids(study)
    coding=o.get('bodySite',{}).get('coding',[])
    codes=[str(c.get('code','')) for c in coding if str(c.get('code','')).startswith('SYN-')]
    if len(codes)!=1:raise ValueError('Ambiguous anatomical source code '+ref)
    match=re.fullmatch(r'SYN-(HIP|ENT)-(LEFT|RIGHT)',codes[0])
    if not match:raise ValueError('Unsupported anatomy code: '+codes[0])
    region={'HIP':'hippocampus','ENT':'entorhinal'}[match.group(1)]
    side=match.group(2).lower()
    p=prov_by_target.get(ref,[])
    if len(p)!=1:raise ValueError('Expected one observation-level Provenance: '+ref)
    p_ref,p_resource=p[0]
    p_sources=sorted({e.get('what',{}).get('reference','') for e in p_resource.get('entity',[]) if e.get('role')=='source'})
    p_sources=[x for x in p_sources if x]
    device_ref=o.get('device',{}).get('reference','')
    if not device_ref or by_ref.get(device_ref,{}).get('resourceType')!='Device':
        raise ValueError('Missing device or unresolved Device: '+ref)
    if study_ref not in p_sources or device_ref not in p_sources:
        raise ValueError('Missing exact Provenance study/device entity links: '+ref)
    q=o.get('valueQuantity',{})
    if not isinstance(q.get('value'),(int,float)) or isinstance(q.get('value'),bool):
        raise ValueError('Missing source numeric measurement: '+ref)
    raw_value=canonical(q['value'])
    return {
      'observation_id':hashlib.sha256(('ICHI2027-34-opaque:'+o['id']).encode('utf-8')).hexdigest()[:24],
      'source_observation_ref':ref,
      'person_ref':person,
      'visit':source_ext(o,'timepoint-label'),
      'effective_datetime':o.get('effectiveDateTime',''),
      'value_raw':raw_value,
      'study_ref':study_ref,
      'study_uid':study_uid,
      'series_uid':series_uid,
      'study_started':study.get('started',''),
      'modality':modality,
      'device_ref':device_ref,
      'region':region,
      'side':side,
      'unit':q.get('code') or q.get('unit') or '',
      'trace_state':source_ext(o,'trace-state'),
      'trace_rule':source_ext(o,'trace-rule-version'),
      'provenance_id':p_ref,
      'provenance_sources':p_sources,
    }
SOURCE=[evidence_row(ref,o) for ref,o in sorted(obs.items())]
assert len(SOURCE)==148 and len({r['observation_id'] for r in SOURCE})==148
assert len({r['person_ref'] for r in SOURCE})==19 and len(studies)==38
assert set(x['person_ref'] for x in SOURCE)==set(x['synthetic_person_ref'] for x in P33_COHORT)
print(f'Canonical source: {len(SOURCE)} observations, {len(patients)} synthetic people, {len(studies)} studies.')

Canonical source: 148 observations, 19 synthetic people, 38 studies.


## 2. Candidate encoding plans and optimization
The **algorithmic proposal** enumerates admissible source-to-shadow context carriers. A plan is feasible only when the required source functional dependencies hold and **every selected evidence field can be reconstructed exactly**. The objective is the smallest *actual canonical JSON byte length for the metadata carrier*; the unchanged numeric base table is not counted in any candidate.

Plans: `INLINE` (full per-row metadata); `STUDY` (shared per-study dictionary); `VISIT` (shared per-person/visit dictionary, valid only with exactly one study context per visit); and coded versions of the two dictionary plans (`STUDY_CODED`, `VISIT_CODED`) that also encode repeated anatomy, TRACE and unit values. Failed reconstruction disqualifies a plan. This is **an experimental transport representation, not a claim that any codec is novel**.

In [4]:
BASE_FIELDS=['observation_id','person_ref','visit','effective_datetime','value_raw']
GROUP_FIELDS=['study_ref','study_uid','series_uid','study_started','modality','device_ref']
LOCAL_FIELDS=['source_observation_ref','region','side','unit','trace_state','trace_rule','provenance_id','provenance_sources']
CONTEXT_FIELDS=GROUP_FIELDS+LOCAL_FIELDS
assert sorted(set(BASE_FIELDS+CONTEXT_FIELDS))==sorted(SOURCE[0].keys())
def context(r):return {k:copy.deepcopy(r[k]) for k in CONTEXT_FIELDS}
def base_table(source):return [{k:copy.deepcopy(r[k]) for k in BASE_FIELDS} for r in source]
def visit_key(row):return canonical([row['person_ref'],row['visit']])
class PlanError(ValueError):pass

def make_carrier(source,plan):
    if plan not in ['INLINE','STUDY','VISIT','STUDY_CODED','VISIT_CODED']:
        raise PlanError('Unknown plan')
    use_study=plan.startswith('STUDY')
    use_visit=plan.startswith('VISIT')
    coded=plan.endswith('_CODED')
    source=sorted(source,key=lambda x:x['observation_id'])
    if len({r['observation_id'] for r in source})!=len(source):raise PlanError('duplicate observation')
    payload={'format':'ICHI2027_PHASE34_ALPHA1','plan':plan,'row_context':{},'group_context':{},'codebooks':{}}
    groups={}
    if use_study or use_visit:
        for r in source:
            key=r['study_ref'] if use_study else visit_key(r)
            g={k:copy.deepcopy(r[k]) for k in GROUP_FIELDS}
            if key in groups and groups[key]!=g:
                raise PlanError('Unsafe inferred source functional dependency for '+key)
            groups[key]=g
        payload['group_context']={k:groups[k] for k in sorted(groups)}
    if coded:
        anatomy=sorted({canonical([r['region'],r['side']]) for r in source})
        traces=sorted({canonical([r['trace_state'],r['trace_rule']]) for r in source})
        a_index={key:f'A{i:03d}' for i,key in enumerate(anatomy)}
        t_index={key:f'T{i:03d}' for i,key in enumerate(traces)}
        units={r['unit'] for r in source}
        one_unit=next(iter(units)) if len(units)==1 else None
        payload['codebooks']={
            'anatomy':{a_index[k]:json.loads(k) for k in anatomy},
            'trace':{t_index[k]:json.loads(k) for k in traces},
            'global_unit':one_unit}
    for r in source:
        if plan=='INLINE':
            row_ctx=context(r)
        else:
            row_ctx={k:copy.deepcopy(r[k]) for k in LOCAL_FIELDS}
            if use_study:row_ctx['study_ref']=r['study_ref']
            if coded:
                row_ctx['anatomy_code']=a_index[canonical([r['region'],r['side']])]
                row_ctx['trace_code']=t_index[canonical([r['trace_state'],r['trace_rule']])]
                for key in ['region','side','trace_state','trace_rule']:del row_ctx[key]
                if one_unit is not None:del row_ctx['unit']
        payload['row_context'][r['observation_id']]=row_ctx
    return payload

def reconstruct(base,carrier):
    plan=carrier.get('plan','')
    if carrier.get('format')!='ICHI2027_PHASE34_ALPHA1':raise PlanError('Wrong carrier format')
    rows=carrier['row_context'];groups=carrier['group_context'];codes=carrier['codebooks']
    if set(rows)!={r['observation_id'] for r in base}:raise PlanError('Missing or extra row context')
    reconstructed=[]
    for b in base:
        rowctx=copy.deepcopy(rows[b['observation_id']]);r=copy.deepcopy(b)
        if plan=='INLINE':
            r.update(rowctx)
        else:
            if plan.endswith('_CODED'):
                try:
                    r['region'],r['side']=codes['anatomy'][rowctx.pop('anatomy_code')]
                    r['trace_state'],r['trace_rule']=codes['trace'][rowctx.pop('trace_code')]
                    if codes['global_unit'] is not None:r['unit']=codes['global_unit']
                except (KeyError,IndexError,TypeError) as exc:raise PlanError('Invalid/missing codebook entry') from exc
            if plan.startswith('STUDY'):
                key=rowctx['study_ref']
            elif plan.startswith('VISIT'):
                key=visit_key(b)
            else:raise PlanError('Unknown carrier plan')
            try:r.update(groups[key])
            except KeyError as exc:raise PlanError('Unknown group key') from exc
            r.update(rowctx)
        if set(r)!=set(BASE_FIELDS+CONTEXT_FIELDS):
            raise PlanError(f'Missing/unexpected evidence fields {b["observation_id"]}: {sorted(set(r)^set(BASE_FIELDS+CONTEXT_FIELDS))}')
        reconstructed.append(r)
    return sorted(reconstructed,key=lambda x:x['observation_id'])

BASE=base_table(SOURCE)
SOURCE_CANONICAL=canonical(sorted(SOURCE,key=lambda x:x['observation_id']))
SOURCE_CANONICAL_SHA256=hashlib.sha256(SOURCE_CANONICAL.encode('utf-8')).hexdigest()
def exact_recovery(candidate,base=BASE,truth=SOURCE):
    restored=reconstruct(base,candidate)
    if canonical(restored)!=canonical(sorted(truth,key=lambda x:x['observation_id'])):
        raise PlanError('Exact field reconstruction failed')
    return restored
PLANS=['INLINE','STUDY','VISIT','STUDY_CODED','VISIT_CODED']
CARRIERS={}; PLAN_AUDIT=[]
for name in PLANS:
    try:
        carrier=make_carrier(SOURCE,name)
        exact_recovery(carrier)
        size=byte_length(carrier)
        CARRIERS[name]=carrier
        PLAN_AUDIT.append({'plan':name,'admissible':True,'exact_reconstruction':True,
                           'metadata_bytes':size,'shared_group_count':len(carrier['group_context']),
                           'anatomy_code_count':len(carrier['codebooks'].get('anatomy',{})),
                           'trace_code_count':len(carrier['codebooks'].get('trace',{})),
                           'reason':'VALID_SYNTHETIC_CANONICAL_FIELDS'})
    except (PlanError,KeyError,TypeError) as exc:
        PLAN_AUDIT.append({'plan':name,'admissible':False,'exact_reconstruction':False,
                           'metadata_bytes':'','shared_group_count':'','anatomy_code_count':'',
                           'trace_code_count':'','reason':str(exc)})
assert 'INLINE' in CARRIERS and len(CARRIERS)>=2
BEST_NAME=min(CARRIERS,key=lambda name:(byte_length(CARRIERS[name]),name))
BEST=CARRIERS[BEST_NAME]
RECOVERED=exact_recovery(BEST)
INLINE_BYTES=byte_length(CARRIERS['INLINE'])
BEST_BYTES=byte_length(BEST)
SAVING=round(100*(INLINE_BYTES-BEST_BYTES)/INLINE_BYTES,4)
print('Selected adaptive carrier:',BEST_NAME)
print('Inline metadata bytes:',INLINE_BYTES,'selected metadata bytes:',BEST_BYTES,
      'reduction vs inline:',SAVING,'%')
print('All plans:',[(r['plan'],r['metadata_bytes']) for r in PLAN_AUDIT])

Selected adaptive carrier: VISIT_CODED
Inline metadata bytes: 92435 selected metadata bytes: 56408 reduction vs inline: 38.9755 %
All plans: [('INLINE', 92435), ('STUDY', 72529), ('VISIT', 65524), ('STUDY_CODED', 63413), ('VISIT_CODED', 56408)]


## 3. Independent structural and failure-closed controls
An additional deliberately ambiguous synthetic study within a visit must make the `VISIT` carrier **inadmissible**, so that the algorithm never infers an imaging study solely from timepoint when two studies coexist. Removal or tampering of the chosen carrier must **fail the exact recovery test**. This tests source-fidelity properties only, not clinical accuracy.

In [5]:
TESTS=[]
def run_test(name,fn):
    try:
        fn();TESTS.append({'test':name,'passed':True,'result':'PASS'})
    except Exception as e:
        TESTS.append({'test':name,'passed':False,'result':repr(e)})

def test_all_admissible_recover():
    for carrier in CARRIERS.values():exact_recovery(carrier)

def test_ambiguous_visit_rejected():
    mutated=copy.deepcopy(SOURCE)
    extra=copy.deepcopy(mutated[0]);extra['observation_id']='augmented-ambiguous-study-test'
    extra['study_ref']='ImagingStudy/synthetic-second-same-visit'
    extra['study_uid']='2.25.77777777777777777777777777777777777777'
    extra['series_uid']='2.25.88888888888888888888888888888888888888'
    extra['provenance_id']='Provenance/synthetic-second-same-visit'
    extra['provenance_sources']=[extra['study_ref'],extra['device_ref']]
    mutated.append(extra)
    for plan in ['VISIT','VISIT_CODED']:
        try:make_carrier(mutated,plan)
        except PlanError:pass
        else:raise AssertionError(plan+' made unsafe visit-only inference')
    for plan in ['STUDY','STUDY_CODED']:
        c=make_carrier(mutated,plan)
        exact_recovery(c,base_table(mutated),mutated)

def test_missing_row_refused():
    mutated=copy.deepcopy(BEST)
    del mutated['row_context'][SOURCE[0]['observation_id']]
    try:reconstruct(BASE,mutated)
    except PlanError:pass
    else:raise AssertionError('Missing sidecar silently accepted')

def test_context_tampering_detected():
    mutated=copy.deepcopy(BEST)
    first=SOURCE[0]['observation_id']
    mutated['row_context'][first]['provenance_id']='Provenance/unsupported-tamper'
    try:exact_recovery(mutated)
    except PlanError:pass
    else:raise AssertionError('Changed provenance passed exact source recovery')

def test_dictionary_corruption_detected():
    mutated=copy.deepcopy(BEST)
    if mutated['group_context']:
        k=next(iter(mutated['group_context']))
        mutated['group_context'][k]['study_uid']='2.25.9999'
    else:
        mutated['row_context'][SOURCE[0]['observation_id']]['study_uid']='2.25.9999'
    try:exact_recovery(mutated)
    except PlanError:pass
    else:raise AssertionError('Changed study UID passed source recovery')

def test_deterministic_output():
    c=make_carrier(SOURCE,BEST_NAME)
    assert canonical(c)==canonical(BEST)
    assert canonical(reconstruct(BASE,c))==SOURCE_CANONICAL

def test_all_source_prov_exact():
    assert all(r['study_ref'] in r['provenance_sources'] and
               r['device_ref'] in r['provenance_sources'] for r in SOURCE)

def test_value_only_deficiency():
    assert all('region' not in b and 'provenance_id' not in b for b in BASE)

def test_study_context_uniqueness():
    by_study=defaultdict(set)
    for r in SOURCE:
        by_study[r['study_ref']].add(canonical([r[k] for k in GROUP_FIELDS]))
    assert len(by_study)==38 and all(len(v)==1 for v in by_study.values())

for name,func in [
  ('all_admissible_plans_roundtrip_148',test_all_admissible_recover),
  ('same_visit_two_studies_forces_safer_fallback',test_ambiguous_visit_rejected),
  ('missing_envelope_entry_fails_closed',test_missing_row_refused),
  ('provenance_tamper_rejected',test_context_tampering_detected),
  ('study_UID_tamper_rejected',test_dictionary_corruption_detected),
  ('deterministic_byte_identical_replay',test_deterministic_output),
  ('source_provenance_study_and_device_present',test_all_source_prov_exact),
  ('value_only_omits_required_context_by_design',test_value_only_deficiency),
  ('study_functional_dependency_attested_on_fixture',test_study_context_uniqueness),
]:run_test(name,func)
if not all(t['passed'] for t in TESTS):raise RuntimeError('Synthetic safety tests failed: '+str(TESTS))
print(f'PASS: {len(TESTS)}/{len(TESTS)} synthetic transport controls.')

PASS: 9/9 synthetic transport controls.


## 4. Pre-specified synthetic cohort-query preservation
A **source-proxy bilateral hippocampal completeness** query requires a baseline and a 12-month observation of each hemisphere. The stricter *evidence-qualified* version additionally requires that each record explicitly reports `trace_state=PASS`, matching Study/Device Provenance references, a valid synthetic Study/Series UID, and numeric values with `mm3` units.

The deliberately naive value-only proxy counts at least two numbers per visit. Its inclusion is **not** qualified cohort membership. Since value-only records lack region and provenance, the strict query returns **UNDETERMINED**, not a made-up negative result. Exact source-vs-restored agreement is meaningful **only for this synthetic shadow experiment**.

In [6]:
from collections import defaultdict

def policy_by_person(rows):
    grp=defaultdict(list)
    for r in rows:grp[r['person_ref']].append(r)
    ans={}
    for person,events in sorted(grp.items()):
        have=[];evidence_complete=True
        for visit in ['baseline','12-month']:
            for side in ['left','right']:
                matching=[r for r in events if r['visit']==visit and r['region']=='hippocampus' and r['side']==side]
                have.append(len(matching)==1)
                for r in matching:
                    evidence_complete &= (
                        r['trace_state']=='PASS' and r['unit']=='mm3' and
                        bool(r['study_uid']) and bool(r['series_uid']) and
                        r['study_ref'] in r['provenance_sources'] and
                        r['device_ref'] in r['provenance_sources'])
        ans[person]={'source_proxy_positive':all(have),
                     'evidence_qualified_positive':all(have) and bool(evidence_complete)}
    return ans

source_policy=policy_by_person(SOURCE)
restored_policy=policy_by_person(RECOVERED)
assert source_policy==restored_policy,'Carrier changed a prespecified query'
reported={r['synthetic_person_ref']:r['source_proxy_member'].strip().lower()=='true' for r in P33_COHORT}
assert set(reported)==set(source_policy)
assert all(reported[p]==d['source_proxy_positive'] for p,d in source_policy.items()),    'Independent Phase33 source proxy comparison disagrees; do not make a cohort claim'
base_by_person=defaultdict(list)
for b in BASE:base_by_person[b['person_ref']].append(b)
COHORT=[]
for person,source in source_policy.items():
    b=base_by_person[person]
    naive=all(sum(x['visit']==v and bool(x['value_raw']) for x in b)>=2
              for v in ['baseline','12-month'])
    COHORT.append({
        'synthetic_person_ref':person,
        'phase33_source_proxy_member':reported[person],
        'source_proxy_member':source['source_proxy_positive'],
        'source_evidence_qualified_member':source['evidence_qualified_positive'],
        'restored_source_proxy_member':restored_policy[person]['source_proxy_positive'],
        'restored_evidence_qualified_member':restored_policy[person]['evidence_qualified_positive'],
        'naive_value_only_numeric_proxy_member':naive,
        'value_only_strict_evidence_status':'UNDETERMINED_MISSING_ANATOMY_AND_PROVENANCE',
        'official_target_cohort_equivalence':False,
        'MCI_status':'NOT_ATTESTED',
    })
assert len(COHORT)==19 and all(x['source_proxy_member']==x['restored_source_proxy_member'] for x in COHORT)
print('SOURCE vs RESTORED synthetic proxy agreements:',len(COHORT),'/',len(COHORT))
print('Source proxy members:',sum(x['source_proxy_member'] for x in COHORT))
print('Naive value-only proxy false inclusions:',sum(x['naive_value_only_numeric_proxy_member'] and not x['source_proxy_member'] for x in COHORT))

SOURCE vs RESTORED synthetic proxy agreements: 19 / 19
Source proxy members: 15
Naive value-only proxy false inclusions: 4


## 5. Save public-safe, hash-pinned outputs to the same Drive directory
Files are **strictly synthetic**, including the reconstructed evidence carrier. Keep actual Athena/ADNI material in separately authorized private storage. The original Phase 33 files are not overwritten. Outputs remain labeled *synthetic shadow results* until a real target ETL is completed.

In [7]:
OUT={
 'shareable':OUT_DIR/'ICHI2027_Phase34_SHAREABLE.json',
 'source':OUT_DIR/'ICHI2027_Phase34_CANONICAL_SOURCE_MANIFEST.csv',
 'base':OUT_DIR/'ICHI2027_Phase34_VALUE_ONLY_SHADOW.csv',
 'carrier':OUT_DIR/'ICHI2027_Phase34_ADAPTIVE_EVIDENCE_CARRIER.json',
 'plans':OUT_DIR/'ICHI2027_Phase34_ENCODING_PLAN_AUDIT.csv',
 'tests':OUT_DIR/'ICHI2027_Phase34_RECONSTRUCTION_TESTS.csv',
 'cohort':OUT_DIR/'ICHI2027_Phase34_SOURCE_RESTORED_PROXY_COMPARISON.csv',
 'blockers':OUT_DIR/'ICHI2027_Phase34_RESEARCH_GATES.csv',
}
def write_csv(path,rows,fieldnames=None):
    fields=fieldnames or list(rows[0].keys())
    with path.open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader()
        for r in rows:
            prepared={k:(canonical(v) if isinstance(v,(list,dict)) else v) for k,v in r.items()}
            w.writerow(prepared)
write_csv(OUT['source'],SOURCE)
write_csv(OUT['base'],BASE)
OUT['carrier'].write_text(json.dumps(BEST,indent=2,ensure_ascii=False),encoding='utf-8')
write_csv(OUT['plans'],PLAN_AUDIT)
write_csv(OUT['tests'],TESTS)
write_csv(OUT['cohort'],COHORT)
GATES=[
 {'gate':'phase33_exact_upstream_sha256','status':'VERIFIED','notes':'Matched complete Phase33 same-folder hashes'},
 {'gate':'transport_candidate_reconstruction','status':'PASSED_SYNTHETIC_SHADOW','notes':f'{len(SOURCE)} source records, selected plan '+BEST_NAME},
 {'gate':'phase33_source_proxy_recovered_equivalence','status':'PASSED_SYNTHETIC_SHADOW','notes':'19/19 person-level source-proxy queries preserved'},
 {'gate':'frozen_private_athena_vocabulary','status':'NOT_PROVIDED','notes':'No concepts selected or approved'},
 {'gate':'independently_reviewed_original_19_bindings','status':'NOT_COMPLETED','notes':'36 original required vocabulary gaps remain'},
 {'gate':'faithful_armA_micdm_mapping','status':'NOT_EXECUTED','notes':'This notebook evaluates a simulated value-only representation only'},
 {'gate':'original_generic_armC','status':'NONPORTABLE_NOT_EXECUTED','notes':'Original frozen encounter producer is not rewritten'},
 {'gate':'armD_official_micdm_implementation','status':'PROTOTYPE_ONLY','notes':'Adaptive evidence carrier is a candidate for future imaging-specific Arm D'},
 {'gate':'pyomop_official_target_loading','status':'NOT_EXECUTED','notes':'No official MI-CDM load'},
 {'gate':'real_MRI_validation','status':'NOT_EXECUTED','notes':'Only developer-generated synthetic data'},
 {'gate':'MCI_or_official_OMOP_target_cohort','status':'NOT_TESTED','notes':'Source-proxy query is not a clinical MCI cohort'},
 {'gate':'independent_external_blinded_validation','status':'NOT_COMPLETED','notes':'Current inputs remain developer-generated'},
]
write_csv(OUT['blockers'],GATES)
summary={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'34_adaptive_synthetic_evidence_carrier_prototype',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'privacy':'Public synthetic-only; no ADNI records, no private Athena data, no real DICOM identifiers.',
 'upstream':{'phase33_shareable_sha256':sha(P33_PATH),
             'phase33_bundle_sha256':sha(P33_FILES['bundle']),
             'phase33_cases_sha256':sha(P33_FILES['cases']),
             'phase33_cohort_sha256':sha(P33_FILES['cohort']),
             'pinned_imagewg_commit':p33['upstream']['imagewg_commit'],
             'pinned_omop55_commit':p33['upstream']['omop55_commit']},
 'method':{'name':'Adaptive minimal evidence carrier (candidate imaging-specific transport prototype)',
           'status':'ALGORITHMIC_PROTOTYPE_NOT_VALIDATED_NOVELTY',
           'optimization':'minimize canonical JSON metadata bytes among five prespecified admissible plans with exact source-field roundtrip',
           'selected_plan':BEST_NAME,'plans_admissible':len(CARRIERS),
           'inline_metadata_bytes':INLINE_BYTES,'selected_metadata_bytes':BEST_BYTES,
           'metadata_byte_reduction_vs_inline_percent':SAVING,
           'full_source_selected_field_manifest_sha256':SOURCE_CANONICAL_SHA256,
           'clinical_validation':False,'independent_external_blinding':False},
 'synthetic_fixture':{'people':19,'imaging_studies':38,'observations':148,
                      'source_only_procedure_resources':sum(r['resourceType']=='Procedure' for r in RESOURCES),
                      'selected_source_evidence_field_count':len(BASE_FIELDS+CONTEXT_FIELDS),
                      'value_only_observation_keys_are_opaque':True,
                      'original_phase33_injected_labels_used_to_fit_method':False},
 'tests':{'synthetic_transport_test_total':len(TESTS),'synthetic_transport_tests_passing':sum(t['passed'] for t in TESTS),
          'exact_selected_field_roundtrip_observations':len(RECOVERED),
          'source_vs_restored_proxy_query_agreement_people':len(COHORT),
          'source_proxy_positive_people':sum(x['source_proxy_member'] for x in COHORT),
          'value_only_naive_false_inclusions_vs_source_proxy':sum(x['naive_value_only_numeric_proxy_member'] and not x['source_proxy_member'] for x in COHORT),
          'real_patient_or_MCI_inference':False},
 'unresolved':{'frozen_athena_snapshot_supplied':False,'original_binding_decisions_reviewed':0,
               'original_vocabulary_gaps_blocked':36,'synthetic_faithful_armA_completed':False,
               'original_generic_armC_executed':False,'official_MiCDM_loading':False,
               'real_MRI_rows_processed':0,'true_four_arm_comparison_completed':False,
               'official_target_cohort_equivalence_tested':False},
 'artifact_sha256':{k:sha(path) for k,path in OUT.items() if k!='shareable'},
 'next_stage_gate':'Define and pre-register an imaging-specific transport experiment against a standards-compliant Arm A after frozen Athena and source-supported binding review; use independent blinded cases and real authorized MRI before clinical claims.'
}
OUT['shareable'].write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('Saved Phase34 outputs:')
for k,p in OUT.items():print(' ',k,p.name)

Saved Phase34 outputs:
  shareable ICHI2027_Phase34_SHAREABLE.json
  source ICHI2027_Phase34_CANONICAL_SOURCE_MANIFEST.csv
  base ICHI2027_Phase34_VALUE_ONLY_SHADOW.csv
  carrier ICHI2027_Phase34_ADAPTIVE_EVIDENCE_CARRIER.json
  plans ICHI2027_Phase34_ENCODING_PLAN_AUDIT.csv
  tests ICHI2027_Phase34_RECONSTRUCTION_TESTS.csv
  cohort ICHI2027_Phase34_SOURCE_RESTORED_PROXY_COMPARISON.csv
  blockers ICHI2027_Phase34_RESEARCH_GATES.csv


In [8]:
# Final exact Drive read-back, not a claim of official MI-CDM/OMOP execution.
check=json.loads(OUT['shareable'].read_text(encoding='utf-8'))
for key,d in check['artifact_sha256'].items():
    assert sha(OUT[key])==d,(key,'Drive read-back SHA mismatch')
assert len(load_csv(OUT['source']))==148
assert len(load_csv(OUT['cohort']))==19
assert all(x['passed'] for x in TESTS)
assert check['unresolved']['official_MiCDM_loading'] is False
assert check['unresolved']['original_vocabulary_gaps_blocked']==36
assert exact_recovery(json.loads(OUT['carrier'].read_text(encoding='utf-8')))==RECOVERED
print('PHASE 34 SYNTHETIC PROTOTYPE COMPLETE — outputs verified in Drive.')
print('For the next notebook, upload:')
for k in ['shareable','plans','tests','cohort']:print(' -',OUT[k].name)

PHASE 34 SYNTHETIC PROTOTYPE COMPLETE — outputs verified in Drive.
For the next notebook, upload:
 - ICHI2027_Phase34_SHAREABLE.json
 - ICHI2027_Phase34_ENCODING_PLAN_AUDIT.csv
 - ICHI2027_Phase34_RECONSTRUCTION_TESTS.csv
 - ICHI2027_Phase34_SOURCE_RESTORED_PROXY_COMPARISON.csv


### After running
Share these **public-safe** outputs: `ICHI2027_Phase34_SHAREABLE.json`, `ICHI2027_Phase34_ENCODING_PLAN_AUDIT.csv`, `ICHI2027_Phase34_RECONSTRUCTION_TESTS.csv` and `ICHI2027_Phase34_SOURCE_RESTORED_PROXY_COMPARISON.csv`.

**Interpretation:** Exact reconstruction and metadata compression are demonstrated for the prespecified **synthetic selected fields only**. Next, the carrier requires independently authored evaluation, approved Athena vocabulary, a standards-compliant Arm A comparator, actual OMOP/MI-CDM loading and eligible real MRI data. This notebook does not change the original Phase 33 or earlier files.